In [270]:
'''
Qs: 
Unstocked Forest Qs:
What is an approporiate amount of time for n_years_unstocked_forest? 
If something starts out as short veg should we consider that Forest or Grassland depending on other circumstances? (i.e. if driver is logging)
- Check short veg with open woodlands in Africa to see if areas that should be forest are not being classified correctly? 

Is the logging concessions data rasterized? What are the raster values (i.e. 0/1)?
- Canada: 2016; 
- Equatorial Guinea: 2013; 
- Indonesia: 2021; 
- Liberia: 2016; 
- Malaysia: 2010; 
- Republic of the Congo: 2013
- Otherwise: unknown


Shifting Cultivation Qs:
Proposed rules said to classify as "Cropland" if mix of short veg/ tall veg and driver == Shifting cultivation. 
- Instead, if it starts out as forest, assume forest until short veg or cropland occurs. Once short veg or cropland occurs, assume cropland for the rest of the timeseries if driver == Shifting cultivation? 
- Otherwise, if it starts out as short veg, assume cropland for the whole time series if driver ==  Shifting cultivation. 
Proposed rule said to classify as "Cropland" + driver == Shifting cultivation. Do we want to extend "Cropland" classification in the timeseries when driver != Shifting cultivation?
Should we use n_years_fallow_cropland to determine transition from cropland back to forest? 


Cautions: 
- No shifting cultivation establishment year. So assuming Cropland the first time tall --> short veg or after "Cropland" LC class.

'''

import re
import pandas as pd
import numpy as np

In [271]:
n_years_unstocked_forest = 3    # number of years forest is allowed to be unstocked before being considered a forest --> grassland conversion

years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

lc_code_map = {
    1: "built up", 
    2: "cropland", 
    3: "tall veg", 
    4: "short veg", 
    5: "wetland", 
    6: "bare",
    7: "water",
    8: "snow/ice",
}

driver_code_map = {
    1: "Permanent agriculture",
    2: "Hard commodities",
    3: "Shifting cultivation",
    4: "Logging",
    5: "Wildfire",
    6: "Settlements & infrastructure",
    7: "Other natural disturbances"
}

sdpt_type_code_map = {
    1: "planted forest",
    2: "tree crop"
}

sdpt_name_code_map = {
    1: "oil palm",
    2: "wood fiber",
    3: "other"
}


In [272]:
# Default LC numeric values -> LU classes 
forest_lc = {3, 7}              # Tall vegetation
grass_lc = {1, 2, 5, 6}         # Short veg
cropland_lc = {10}              # Cropland
settlement_lc = {11}            # Built up
wetland_lc = {4}                # Wetland
other_lc = {0, 8, 9}            # Bare, water, snow/ice

# Lookup table from LC code -> token
lc_token_map = {
    **{v: "F" for v in forest_lc},
    **{v: "G" for v in grass_lc},
    **{v: "C" for v in cropland_lc},
    **{v: "S" for v in settlement_lc},
    **{v: "W" for v in wetland_lc},
    **{v: "O" for v in other_lc},
}

# Function to get land use token per land cover numeric value (tokens used for regex exception rules)
def token_for_lc(v):
    return lc_token_map.get(v, "-")

In [273]:
# Exceptions to the default land cover to land use assignments
#TODO: Consider including "other" LU class in these exceptions
#TODO: Add in JRC NGHGI conversion rules
#TODO: Add in GMWv3 to temp water/flooded land exceptions

# # If pixel is in a logging concessions area, assume any short vegetation is unstocked forest so land use remains "Forest Land"
# def apply_logging_concessions(ctx):
#     if ctx["logging_concession"] == 1:
#         for i, lc in enumerate(ctx["tokens"]):
#             if lc == "G":
#                 ctx["lu_vals"][i] = "Forest land"
# TODO: Decided not to do this because there is a fair amount of grassland in logging concessions, instead using TCL to determine if it is unstocked forest

# If pixel is in SDPT planted forest, assume any lower hierarchy class (i.e. short veg or bare) is unstocked forest so land use remains "Forest Land"
def apply_sdpt_planted_forest(ctx):
    if ctx["sdpt_type"] == 1:
        for i, lc in enumerate(ctx["tokens"]):
            if lc == "G":
                ctx["lu_vals"][i] = "Forest land"
                
# If timeseries switches between short veg and tall veg and duration of short veg does not exceed n_years_unstocked_forest
# assume temporarily unstocked forest so land use remains "Forest Land"
def apply_temp_unstocked_forest(ctx):
    # If timeseries starts as short veg but switches to tall veg before n_years_unstocked_forest (i.e start -> G -> F)
    pattern = rf"^(G{{1,{ctx['n_unstocked']}}})F"
    m = re.search(pattern, ctx["token_seq"])
    if m:
        a, b = m.span(1)
        for i in range(a, b):
            ctx["lu_vals"][i] = "Forest land" 
    
    # If tall veg switches to short veg for less than n_years_unstocked_forest and returns back to tall veg (i.e F -> G -> F)
    pattern = rf"F(G{{1,{ctx['n_unstocked']}}})(?=F)"   # Note: ?=Positive look ahead for following F so it's not included in match group (i.e. FGFGFGFGF)
    for m in re.finditer(pattern, ctx["token_seq"]):
        if m:
            a, b = m.span(1)
            for i in range(a, b):
                ctx["lu_vals"][i] = "Forest land"
    
    # If timeseries switches from tall veg and ends with short veg before n_years_unstocked_forest (i.e. F -> G -> end)
    pattern = rf"F(G{{1,{ctx['n_unstocked']}}})$"
    m = re.search(pattern, ctx["token_seq"])
    if m:
        a, b = m.span(1)
        for i in range(a, b):
            ctx["lu_vals"][i] = "Forest land" 

# If the driver is "logging" and the timeseries starts as short veg followed by tall veg or switches from tall veg to short veg until the end of the timeseries 
# assume unstocked forest so land use remains "Forest Land"
# Note: short veg not limited to n_years_unstocked_forest here
# TODO: What about interior short veg when the driver is logging? Should we apply similar rule as SDPT and logging concessions (i.e. all short veg gets reclassified)? 
def apply_forest_logging(ctx):
    if ctx["driver"] == 4:
        # Starts with consecutive short veg followed by tall veg (regardelss if n_years_unstocked_forest is met)
        pattern = r"^(G+)F"
        m = re.search(pattern, ctx["token_seq"])
        if m:
            a, b = m.span(1) 
            for i in range(a, b):
                ctx["lu_vals"][i] = "Forest land"
        
        # Tall veg followed by consecutive short veg until the end (regardelss if n_years_unstocked_forest is met)
        pattern = r"F(G+)$"
        m = re.search(pattern, ctx["token_seq"])
        if m:
            a, b = m.span(1) 
            for i in range(a, b):
                ctx["lu_vals"][i] = "Forest land"

#TODO: Revisit this rule after GEE analysis. Should we assume starting tall veg is also cropland (i.e. how to determine establishment year)? What about cropland to short veg/ tall veg if driver is not shifting cult? 
# Function to determine fallow cropland if driver is shifting cultivation
# def apply_cropland_shifting_cultivation(ctx):
#     if ctx["driver"] == 3:
#         # If timeseries starts as short veg or cropland and the driver is shifting cultivation, all subsequent short and tall veg is assumed to be fallow cropland so land use is considered "Cropland"
#         pattern = r"^[GC]"
#         m = re.match(pattern, ctx["token_seq"])
#         if m:
#             for i, lu in enumerate(ctx["tokens"]):
#                 if lu in ("F", "G"):
#                     ctx["lu_vals"][i] = "Cropland"
#
#         # Otherwise, if timeseries starts with tall veg but switches to short veg or cropland at some point and the driver is shifting cultivation,
#         # any subsequent short or tall veg is assumed to be fallow cropland so land use is considered "Cropland"
#         # TODO: Update so we don't assume forest to begin with
#         else:
#             pattern = r"^F+([GC].*)$"
#             m = re.match(pattern, ctx["token_seq"])
#             if m:
#                 a, b = m.span(1)
#                 for i in range(a, b):
#                     if ctx["tokens"][i] in ("F", "G"):
#                         ctx["lu_vals"][i] = "Cropland"
# TODO: Decided to consider shifting cultivation Forest

# If sdpt tree crops, reassign land use as "Cropland"
#TODO: Should we make any assumptions about SDPT tree crop establishment year for non oil palm? 
def apply_sdpt_tree_crop(ctx):
    if ctx["sdpt_type"] == 2:
        # If oil palm, all years after Descals establishment year are assumed to be "Cropland"
        if ctx["sdpt_name"] == 1 and ctx["establishment_year"] is not None:
            if ctx["establishment_year"] <= years[0]: 
                 for i, lu in enumerate(ctx["tokens"]):
                    ctx["lu_vals"][i] = "Cropland"
            else: 
                est_index = years.index(ctx["establishment_year"])
                for offset, lu in enumerate(ctx["tokens"][est_index:]):
                    i = est_index + offset
                    ctx["lu_vals"][i] = "Cropland"

        # Otherwise, assume any short veg or tall veg in the entire timeseries is tree crops so land use becomes "Cropland"
        else:
             for i, lu in enumerate(ctx["tokens"]):
                if lu in ("F", "G"):
                    ctx["lu_vals"][i] = "Cropland"

#TODO: Revisit this rule after GEE analysis. Should this apply to the entire timeseries instead of just after W if there is a lot of noise in temp flooded land? 
def apply_temp_flooded(ctx): 
    # TODO: Consider perm flooded (W -> F -> W -> W -> W). Impoundment layers: https://www.globaldamwatch.org/database 
    # Both short veg and wetland get reclassified as forest 
    #pattern = r"^.*?([WO].*F)"    # Only reclassifies between W/O and first F 
    pattern = r"^.*?([WO].*F.*)$"  # Reclassifies everything between W/O and end if F after W/O
    m = re.match(pattern, ctx["token_seq"])
    if m:
        a, b = m.span(1)         
        for i in range(a, b): 
            if ctx["tokens"][i] in ("W", "O", "G"):
                ctx["lu_vals"][i] = "Forest land"

    else: 
         #pattern = r"^.*?([WO].*G)"    # Only reclassifies between W/O and first G, if no F
        pattern = r"^.*?([WO].*G.*)$"  # Reclassifies everything between W/O and end if G after W/O (and no F)
        m = re.match(pattern, ctx["token_seq"])
        if m:
            a, b = m.span(1)         
            for i in range(a, b): 
                if ctx["tokens"][i] in ("W", "O"):
                    ctx["lu_vals"][i] = "Grassland"
                    

                

In [274]:
#TODO: If only allowing for one rule to apply, then need to call terminal funcitons in other exceptions (i.e unstocked forest rule also checks unstocked forest at the end even if LC doesnt return to F, etc)
rules = {"kind": 
        # Default rules to go from land cover to land use have already been applied (via tokens)
        # so just assigning the labels and names for those default assumptions here
        {"default": {
            
                    "F": {"label": "Forest land",
                          "names": "forest_default"},
                    "N": {"label": "Grassland",
                          "names": "grass_default"},
                    "G": {"label": "Grassland",
                          "names": "grass_default"},
                    "C": {"label": "Cropland",
                          "names": "cropland_default"},
                    "S": {"label": "Settlements",
                          "names": "settlements_default"},
                    "W": {"label": "Wetlands",
                          "names": "wetlands_default"},
                    "O": {"label": "Other land",
                          "names": "other_default"},
                    },
        
        # Each exception calls a function which uses regex to see if that condition applies to this lu timeseries
        # If that condition applies, the lu values are updated accordingly and the name is updated with the exception applied
        "exception": [
            
        # Confusion between Forest and Grassland --------------------------------------------------------------------
            
            # Forestry exceptions
            {"name": "forest_logging_concessions",
             "apply": apply_logging_concessions},

            {"name": "forest_sdpt_planted_forest",
             "apply": apply_sdpt_planted_forest},

            {"name": "forest_temp_unstocked_forest",
             "apply": apply_temp_unstocked_forest},

            {"name": "forest_logging_terminal",
             "apply": apply_forest_logging},
            
            
            # Cropland exceptions 
            {"name": "cropland_shifting_cultivation",
             "apply": apply_cropland_shifting_cultivation},
            
            #TODO: Come back to rule #5
            
            {"name": "cropland_sdpt_tree_crop",
             "apply": apply_sdpt_tree_crop},
            
            # Temporary flooded forest or grassland
            {"name": "forest_grass_temp_flood",
             "apply": apply_temp_flooded},

            # {"name": "",
            #  "apply": },
            # 
            # {"name": "",
            #  "apply": },

        ]
    }
}

# Function to apply default rules to go from GLAD land cover to IPCC land Use
def lc_to_lu_default(lc_vals, rules_table):
    lu_vals = []
    for lc in lc_vals:
        try:
            lu = lc_token_map[lc]
            assigned = rules_table["kind"]["default"][lu]["label"]
            lu_vals.append(assigned)
        except KeyError:
            lu_vals.append(None)
    
    return lu_vals

In [275]:
# Iterate through each scenarios and classify LC to create LU timeseries
def classify_dataframe(df, rules, lc_cols):
    out = []
    for idx, row in df.iterrows():
        driver      = int(row["driver"]) if pd.notna(row["driver"]) else None
        sdpt_type   = int(row["sdpt_type"])   if pd.notna(row["sdpt_type"])   else None
        sdpt_name   = int(row["sdpt_name"])   if pd.notna(row["sdpt_name"])   else None
        estab_year  = int(row["establishment_year"])  if pd.notna(row["establishment_year"])   else None
        logc        = int(row["logging_concession"]) if pd.notna(row["logging_concession"]) else None
        lc_ts  = [row.get(c) for c in lc_cols]

        lu_ts = classify_scenario(row["id"], lc_ts, rules, driver, sdpt_type, sdpt_name, estab_year, logc)

        # Create annual land use columns
        lu_cols = {f"lu_{y}": lu_ts[i] for i,y in enumerate(years)}

        # Create time step transitions and conversion flags
        conversion = False
        trans_cols = {}
        for i,(a,b) in enumerate(zip(years[:-1], years[1:])):
            a_lu, b_lu = lu_ts[i], lu_ts[i+1]
            key = f"{a}_{b}"
            if a_lu == b_lu:
                trans_cols[key] = f"{a_lu} remaining {b_lu}"
            else:
                trans_cols[key] = f"{a_lu} to {b_lu}"
                conversion = True

        out.append({
            "id": row["id"],
            "driver": driver_code_map.get(driver, str(driver) if driver is not None else None),
            "sdpt_type":   sdpt_type_code_map.get(sdpt_type,   str(sdpt_type)   if sdpt_type   is not None else None),
            "sdpt_name":   sdpt_name_code_map.get(sdpt_name,   str(sdpt_name)   if sdpt_name   is not None else None),
            "logging_concession": "yes" if logc == 1 else None,
            **lu_cols,
            **trans_cols,
            "conversion_occurred": conversion
        })
        
    return pd.DataFrame(out)

In [276]:
# Main function to apply the regex rules to the LC timeseries and returns the final LU timeseries
def classify_scenario(scenario_id, lc_timeseries, rules, driver=None, sdpt_type=None, sdpt_name=None, establishment_year = None, logging_concession=None):
    
    # Create array with default LU tokens using LC timeseries
    lc_vals = [int(v) for v in lc_timeseries]
    lu_vals = lc_to_lu_default(lc_vals, rules)

    # Build tokens sequence (string with no spaces) and apply exception rules
    tokens = [token_for_lc(v) for v in lc_vals]
    token_seq = "".join(tokens)

    # Inputs for exception rules
    ctx = {
        "scenario_id": scenario_id,
        "lc_vals": lc_vals,
        "lu_vals": lu_vals,
        "tokens": tokens,
        "token_seq": token_seq,
        "driver": driver,
        "sdpt_type": sdpt_type,
        "sdpt_name": sdpt_name,
        "establishment_year": establishment_year,
        "logging_concession": logging_concession,
        "n_unstocked": n_years_unstocked_forest
    }
    
    # Override defaults if exception(s) apply
    for rule in rules["kind"]["exception"]:
        rule["apply"](ctx)
    
    return lu_vals

In [277]:
# Configuration
file_path = "/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/scripts/postprocessing/conversion_LUC_scenarios.xlsx"  # Update if needed
sheet_name = "scenarios"

lc_cols = [f"lc_{y}" for y in years]

In [278]:
# Load data
scenarios_df = pd.read_excel(file_path, sheet_name=sheet_name)

In [279]:
# Run classification
# Coerce scenarios_df to numeric
for c in lc_cols + ["driver", "sdpt", "logging_concession"]:
    if c in scenarios_df.columns: scenarios_df[c] = pd.to_numeric(scenarios_df[c], errors="coerce")
    
# Run classification
results_df = classify_dataframe(scenarios_df, rules, lc_cols)

# Output results 
print("\nClassification Results:")
print(results_df)


Classification Results:
      id                driver       sdpt_type sdpt_name logging_concession  \
0    1.0                  None            None      None                yes   
1    2.0                  None  planted forest      None               None   
2    3.0                  None            None      None               None   
3    4.0                  None            None      None               None   
4    5.0                  None            None      None               None   
5    6.0               Logging            None      None               None   
6    7.0               Logging            None      None               None   
7    8.0                  None            None      None               None   
8    9.0                  None            None      None               None   
9   10.0                  None            None      None               None   
10  11.0                  None            None      None               None   
11  12.0                  N

In [280]:
results_df.to_excel("/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/scripts/postprocessing/conversion_LUC_scenario_results.xlsx", index=False)